# 🧪 Laboratório Prático: Tokenização, Embeddings e Inferência Local em CPU
**Disciplina:** COM170 — Inteligência Artificial na Prática Acadêmica e Profissional  
**Quinzena 02 — Módulo 1:** Prompts, Tokenização e Modelos Generativos  
**Perfil:** Estudo Prático Guiado (Passo a Passo)

---

## 🎯 Objetivos de Aprendizagem
Neste laboratório, você verá na prática o que acontece "por baixo do capô" de um modelo de linguagem (LLM):

1. **Tokenização:** Como o texto humano é fragmentado em unidades menores (tokens) e mapeado para números inteiros (*Token IDs*).
2. **Embeddings:** Como esses IDs são convertidos em vetores numéricos contínuos de alta dimensionalidade ($d_{model} = 768$ no GPT-2) que capturam relações semânticas.
3. **Inferência Local e Autoregressiva:** Como um modelo ultracompacto moderno (**Qwen 2.5 0.5B Instruct**) é carregado e gera texto token a token em tempo real na CPU do seu próprio computador.

---

## ⚙️ Pré-requisitos: Isolamento com Ambiente Virtual (`venv`)
Para manter seu Python global limpo, isole as dependências deste laboratório criando um ambiente virtual:

### 1. Criar e Ativar o Ambiente Virtual
Abra o terminal nesta pasta e execute:
```bash
# Criar a pasta do ambiente virtual (.venv)
python -m venv .venv

# Ativar o ambiente virtual:
# Windows (PowerShell):
.venv\Scripts\Activate.ps1
# Windows (Prompt de Comando CMD):
.venv\Scripts\activate.bat
# Linux / macOS:
source .venv/bin/activate
```

### 2. Instalar as Bibliotecas e Suporte ao Jupyter (`ipykernel`)
Com o ambiente ativado (você verá `(.venv)` no início da linha de comando):
```bash
pip install --upgrade pip
pip install transformers torch ipykernel
```

### 3. Selecionar o Kernel no VS Code / Jupyter
> 💡 **Dica de execução:** No canto superior direito deste notebook no VS Code, clique em **Select Kernel** (ou *Selecionar Kernel*) $\rightarrow$ **Python Environments...** $\rightarrow$ escolha o interpretador dentro de `.venv`.


## 📦 Etapa 1: Importação das Bibliotecas e Verificação do Ambiente

Vamos importar as classes essenciais da biblioteca `transformers` do Hugging Face e a biblioteca `torch` (PyTorch).

In [ ]:
import time
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    GPT2Tokenizer,
    GPT2Model
)

print(f"Versão do PyTorch: {torch.__version__}")
print(f"Dispositivo disponível para execução: {'CUDA (GPU)' if torch.cuda.is_available() else 'CPU'}")

---
## 🧠 Etapa 2: Investigação com Modelo Didático (GPT-2 Small - ~500 MB)

O **GPT-2 Small** (OpenAI) possui aproximadamente 124 milhões de parâmetros e uma dimensão oculta de embedding de 768. Ele é perfeito para estudo e inspeção de tensores porque é leve, rápido para carregar e seus componentes internos são amplamente documentados.

### 2.1 Carregamento do Tokenizador e do Modelo Base
Aqui carregamos o `GPT2Tokenizer` (que usa o algoritmo BPE — *Byte Pair Encoding*) e o `GPT2Model` (que contém os pesos do modelo e a tabela de embeddings).

In [ ]:
# Carregamento do tokenizador e do modelo didático GPT-2
nome_modelo_didatico = "gpt2"

print("Baixando/Carregando tokenizador e modelo GPT-2...")
tokenizador_didatico = GPT2Tokenizer.from_pretrained(nome_modelo_didatico)
modelo_didatico = GPT2Model.from_pretrained(nome_modelo_didatico)
print("Carregamento concluído com sucesso!")

### 2.2 Tokenização e Inspeção dos Token IDs

Vamos testar o prompt em inglês `"Senior Software Engineering"`.
Observe como o tokenizador decompõe o texto em tokens e os converte em números inteiros (*Token IDs*).

In [ ]:
prompt_ingles = "Senior Software Engineering"

# Codificação do prompt em tensores PyTorch
tokens_identificadores = tokenizador_didatico.encode(prompt_ingles, return_tensors="pt")

print(f"Texto original: '{prompt_ingles}'")
print(f"Lista de Token IDs: {tokens_identificadores.tolist()[0]}")
print(f"Quantidade total de tokens gerados: {tokens_identificadores.shape[1]}")

# Inspeção token a token
print("\nDecomposição detalhada dos tokens:")
for indice_token, token_id in enumerate(tokens_identificadores[0]):
    token_texto = tokenizador_didatico.decode(token_id)
    print(f"  • Token {indice_token + 1}: ID {token_id.item():<6} -> Representação textual: '{token_texto}'")

### 🔬 2.3 Comparação Didática: Inglês vs. Português

Como visto no material teórico da Quinzena 02, o GPT-2 foi treinado predominantemente com textos em inglês. Por isso, seu vocabulário é muito mais eficiente para o inglês do que para o português.

Vamos comparar a quantidade de tokens gerados para a tradução correspondente em português:

In [ ]:
prompt_portugues = "Engenheiro de Software Sênior"

tokens_portugues = tokenizador_didatico.encode(prompt_portugues, return_tensors="pt")

print(f"Texto em Inglês:    '{prompt_ingles}'   -> {tokens_identificadores.shape[1]} tokens")
print(f"Texto em Português: '{prompt_portugues}' -> {tokens_portugues.shape[1]} tokens")

print("\nDecomposição dos tokens em português:")
for indice_token, token_id in enumerate(tokens_portugues[0]):
    token_texto = tokenizador_didatico.decode(token_id)
    print(f"  • Token {indice_token + 1}: ID {token_id.item():<6} -> '{token_texto}'")

### 2.4 Extração e Inspeção da Matriz de Embeddings (`wte`)

No GPT-2, a camada `wte` (*Word Token Embeddings*) funciona como uma tabela de consulta que mapeia cada *Token ID* inteiro para um vetor denso no espaço multidimensional $\mathbb{R}^{768}$.

Vamos inspecionar o formato do tensor resultante e visualizar os primeiros valores numéricos que compõem o significado de cada token:

In [ ]:
# Passagem dos token IDs pela camada de Word Token Embeddings (wte)
matriz_embeddings = modelo_didatico.wte(tokens_identificadores)

print(f"Formato da matriz de embeddings: {list(matriz_embeddings.shape)}")
print("Estrutura do Tensor: [Batch Size = 1, Quantidade de Tokens = 3, Dimensão do Embedding = 768]")

# Exibição de uma amostra dos primeiros 5 valores numéricos do embedding de cada token
print("\nAmostra dos primeiros 5 valores numéricos de cada vetor:")
for indice_token, token_id in enumerate(tokens_identificadores[0]):
    token_texto = tokenizador_didatico.decode(token_id)
    vetor_amostra = matriz_embeddings[0, indice_token, :5].detach().tolist()
    valores_formatados = [round(valor, 4) for valor in vetor_amostra]
    print(f"  • Token '{token_texto}' (ID {token_id.item()}): {valores_formatados} ... (total: 768 dimensões)")

---
## ⚡ Etapa 3: Inferência Local com Modelo Ultracompacto (Qwen 2.5 0.5B Instruct)

Agora vamos executar um modelo de linguagem moderno com capacidade de seguir instruções (*instruct*), contendo **500 milhões de parâmetros** (**Qwen2.5-0.5B-Instruct**), rodando diretamente na **CPU**.

### 3.1 Carregamento do Modelo e Tokenizador
O modelo ocupa aproximadamente ~350 MB a ~500 MB de memória RAM, sendo ideal para testes rápidos locais sem exigir GPU.

In [ ]:
nome_modelo_ultra = "Qwen/Qwen2.5-0.5B-Instruct"

print(f"Baixando/Carregando {nome_modelo_ultra}...")
tokenizador_ultra = AutoTokenizer.from_pretrained(nome_modelo_ultra)
modelo_ultra = AutoModelForCausalLM.from_pretrained(nome_modelo_ultra)
print("Modelo carregado com sucesso na memória!")

### 3.2 Preparação do Prompt e Tokenização de Entrada

Vamos formular uma pergunta direta para testar a capacidade de síntese conceitual do modelo.

In [ ]:
pergunta_estudo = "Explique em uma frase curta o que e um token em IA:"

entradas_tokenizadas = tokenizador_ultra(pergunta_estudo, return_tensors="pt")

print(f"Pergunta enviada: '{pergunta_estudo}'")
print(f"Quantidade de tokens de entrada (Prompt): {entradas_tokenizadas['input_ids'].shape[1]}")

### 3.3 Execução da Inferência e Medição de Desempenho em CPU

Vamos executar a geração autoregressiva com o método `generate()`, limitando a saída em 30 novos tokens (`max_new_tokens=30`).
Mediremos o tempo exato de resposta e calcularemos a taxa de tokens gerados por segundo (tokens/s).

In [ ]:
# Marcação do tempo inicial
tempo_inicial = time.time()

# Geração autoregressiva dos tokens na CPU
with torch.no_grad():
    saidas_geradas = modelo_ultra.generate(
        **entradas_tokenizadas,
        max_new_tokens=30,
        pad_token_id=tokenizador_ultra.eos_token_id
    )

# Cálculo do tempo total decorrido
tempo_total_decorrido = time.time() - tempo_inicial

# Decodificação dos tokens gerados para texto legível
resposta_completa = tokenizador_ultra.decode(saidas_geradas[0], skip_special_tokens=True)

# Cálculo de métricas de desempenho
quantidade_tokens_gerados = saidas_geradas.shape[1] - entradas_tokenizadas['input_ids'].shape[1]
velocidade_tokens_por_segundo = quantidade_tokens_gerados / tempo_total_decorrido if tempo_total_decorrido > 0 else 0

print("==================================================")
print("📊 RESULTADOS DA EXECUÇÃO LOCAL (CPU)")
print("==================================================")
print(f"⏱️ Tempo de execução em CPU: {tempo_total_decorrido:.2f} segundos")
print(f"🚀 Novos tokens gerados: {quantidade_tokens_gerados} tokens")
print(f"⚡ Velocidade média: {velocidade_tokens_por_segundo:.2f} tokens/segundo")
print("--------------------------------------------------")
print("📝 Resposta Completa Gerada:")
print(resposta_completa)
print("==================================================")

---
## 📝 Etapa 4: Roteiro de Reflexão e Fixação Conceitual

Após rodar todas as células deste laboratório, revise os seguintes conceitos-chave:

1. **Inspeção de Embeddings (GPT-2):**
   - A matriz de saída possui formato `[1, 3, 768]`:
     - `1`: Tamanho do lote (*batch size*), representando uma sequência de entrada.
     - `3`: Número de tokens gerados pelo prompt (`"Senior"`, `"Software"`, `"Engineering"`).
     - `768`: Dimensão oculta do vetor de embedding ($d_{model}$). Cada token é posicionado num espaço semântico de 768 dimensões.

2. **Diferença de Eficiência na Tokenização:**
   - Por que o texto em inglês gerou 3 tokens e em português gerou mais tokens?
   - *Explicação:* O vocabulário BPE reflete os dados de treinamento. Palavras inteiras em inglês existem prontas no vocabulário, enquanto palavras em outros idiomas acabam sendo particionadas em subpalavras ou caracteres isolados.

3. **Viabilidade de Inferência Local:**
   - Como um modelo de 500 milhões de parâmetros conseguiu responder com coerência sem consumir gigabytes de GPU?
   - *Explicação:* Modelos ultracompactos (0.5B) são projetados especificamente para tarefas leves, dispositivos embarcados e testes rápidos de CPU com baixíssimo consumo de memória RAM.